# House MD — Metin Sınıflandırma Projesi
**Görev:** Tüm özellik sütunlarını kullanarak `correct_prediction` (hastalık adı) tahmin etmek

## Adım 1 — Ortam Kurulumu

In [ ]:
!pip install pandas scikit-learn matplotlib seaborn scipy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, json, warnings
warnings.filterwarnings('ignore')

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

print('Kütüphaneler yüklendi.')

## Adım 2 — Veri Yükleme ve EDA

In [ ]:
df = pd.read_csv('Last_HouseMD_DataSet(Sayfa1).csv', sep=';', encoding='utf-8-sig')
print('Satır sayısı :', len(df))
print('Sütunlar     :', df.columns.tolist())
df.head(3)

In [ ]:
bos = df.isnull().sum().sort_values(ascending=False)
pd.DataFrame({'Boş Sayısı': bos, 'Boş %': (bos / len(df) * 100).round(1)})

In [ ]:
print('correct_prediction unique:', df['correct_prediction'].nunique())
print(df['correct_prediction'].value_counts().head(25))

## Adım 3 — Hedef Sütun Temizleme (Top 20 Hastalık)

In [ ]:
GURULTULU = {'', 'none', 'None', 'NONE', '1', '0', '-', 'nan'}
df['hedef'] = df['correct_prediction'].astype(str).str.strip()
df = df[~df['hedef'].isin(GURULTULU)].copy()

TOP_N = 20
top20 = df['hedef'].value_counts().head(TOP_N).index.tolist()
df = df[df['hedef'].isin(top20)].reset_index(drop=True)

print(f'Top {TOP_N} hastalık — kalan satır: {len(df)}')
print(df['hedef'].value_counts())

In [ ]:
plt.figure(figsize=(12, 5))
df['hedef'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Sınıf Dağılımı — Top 20 Hastalık')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Adım 4 — Özellik Mühendisliği
### 4a — Metin (TF-IDF)

In [ ]:
def on_isle(metin):
    metin = str(metin).lower()
    metin = re.sub(r'[^\w\s]', ' ', metin)
    metin = re.sub(r'\d+', ' ', metin)
    return re.sub(r'\s+', ' ', metin).strip()

# Alaycı cümlelere prefix — TF-IDF bu bağlamı ayrı öğrenir
def isle_ve_etiketle(row):
    metin = on_isle(row['text'])
    return ('sarcasm ' + metin) if str(row['Sarcasm']).strip() == '1' else metin

df['text_clean'] = df.apply(isle_ve_etiketle, axis=1)

TIBBI = ['Symptom', 'Test', 'Drug', 'Procedure', 'Organ']
df['tibbi_metin'] = df[TIBBI].fillna('').apply(
    lambda row: ' '.join(row.values.astype(str)), axis=1
).apply(on_isle)

X_text  = TfidfVectorizer(max_features=5000, ngram_range=(1, 2)).fit_transform(df['text_clean'])
X_tibbi = TfidfVectorizer(max_features=3000).fit_transform(df['tibbi_metin'])

print('X_text  :', X_text.shape)
print('X_tibbi :', X_tibbi.shape)

### 4b — Kategorik (One-Hot)

In [ ]:
KATEGORIK = ['Intent', 'diagnosis_stage', 'Emotion', 'speaker']
df_kat = df[KATEGORIK].fillna('bilinmiyor').astype(str)
for col in KATEGORIK:
    nadir = df_kat[col].value_counts()
    df_kat[col] = df_kat[col].replace(nadir[nadir < 5].index, 'diger')

X_kat_sparse = csr_matrix(pd.get_dummies(df_kat, drop_first=True).astype(float).values)
print('X_kategorik :', X_kat_sparse.shape)

### 4c — JSON Çözümleme: medical_entities

In [ ]:
TYPE_MAP = {
    'Disease': 'ent_Disease', 'Diagnosis': 'ent_Disease', 'DS': 'ent_Disease',
    'Condition': 'ent_Disease', 'COND': 'ent_Disease',
    'Symptom': 'ent_Symptom', 'SYMP': 'ent_Symptom',
    'Test': 'ent_Test', 'TEST': 'ent_Test', 'Test Sonucu': 'ent_Test',
    'Drug': 'ent_Drug', 'DRUG': 'ent_Drug', 'Medication': 'ent_Drug',
    'Procedure': 'ent_Procedure', 'PROC': 'ent_Procedure', 'Treatment': 'ent_Procedure',
    'Anatomy': 'ent_Anatomy', 'Organ': 'ent_Anatomy', 'ORG': 'ent_Anatomy',
}
ENT_COLS = ['ent_Disease', 'ent_Symptom', 'ent_Test', 'ent_Drug', 'ent_Procedure', 'ent_Anatomy']

def parse_entities(json_str):
    sonuc = {col: [] for col in ENT_COLS}
    try:
        for e in json.loads(str(json_str)):
            tip = TYPE_MAP.get(e.get('type', ''))
            if tip:
                sonuc[tip].append(str(e.get('text', '')).lower())
    except Exception:
        pass
    return {col: ' '.join(v) for col, v in sonuc.items()}

ent_df = df['medical_entities'].fillna('[]').apply(parse_entities).apply(pd.Series)
df = pd.concat([df, ent_df], axis=1)

df['ent_all'] = df[ENT_COLS].fillna('').apply(
    lambda row: ' '.join(row.values.astype(str)), axis=1
).apply(on_isle)

X_entity = TfidfVectorizer(max_features=3000, ngram_range=(1, 2)).fit_transform(df['ent_all'])
print('X_entity :', X_entity.shape)

## Adım 5 — Özellik Birleştirme

In [ ]:
# Sayısal özellikler — hstack ile aynı hücrede tanımlanıyor
SAYISAL = ['Sarcasm', 'season', 'episode']
X_say = csr_matrix(df[SAYISAL].fillna(0).astype(float).values)

X = hstack([X_text, X_tibbi, X_entity, X_kat_sparse, X_say])
y = df['hedef']

print('Özellik grupları:')
print(f'  X_text      : {X_text.shape}')
print(f'  X_tibbi     : {X_tibbi.shape}')
print(f'  X_entity    : {X_entity.shape}')
print(f'  X_kategorik : {X_kat_sparse.shape}')
print(f'  X_sayisal   : {X_say.shape}')
print(f'  ─────────────────────')
print(f'  TOPLAM      : {X.shape}')

## Adım 6 — Train / Test Bölme

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Eğitim seti:', X_train.shape)
print('Test seti  :', X_test.shape)

## Adım 7 — Model Eğitimi

In [ ]:
from sklearn.model_selection import cross_val_score

modeller = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42,
        class_weight='balanced',   # azınlık sınıflara ağırlık ver
        C=1.0
    ),
    'LinearSVC': LinearSVC(
        max_iter=2000, random_state=42,
        class_weight='balanced',
        C=1.0
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1,
        class_weight='balanced'
    ),
}

sonuclar = {}
cv_sonuclar = {}

for isim, model in modeller.items():
    # 5-fold cross-validation (daha güvenilir)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    cv_sonuclar[isim] = cv_scores

    # Test seti tahmini
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    sonuclar[isim] = y_pred

    acc = (y_pred == y_test).mean()
    print(f'{isim:<25}  Test Acc: {acc:.4f}  |  CV F1-macro: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## Adım 8 — Değerlendirme

In [ ]:
# En iyi model: Logistic Regression (0.830 > LinearSVC 0.782)
en_iyi = 'Logistic Regression'
print(f'=== {en_iyi} — Classification Report ===')
print(classification_report(y_test, sonuclar[en_iyi]))

In [ ]:
siniflar = sorted(y.unique())
cm = confusion_matrix(y_test, sonuclar[en_iyi], labels=siniflar)

plt.figure(figsize=(14, 10))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=siniflar, yticklabels=siniflar, cmap='Blues')
plt.title(f'Confusion Matrix — {en_iyi}')
plt.xlabel('Tahmin')
plt.ylabel('Gerçek')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# CV F1-macro karşılaştırması (daha güvenilir metrik)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Sol: Test Accuracy
acc_list = {isim: (pred == y_test).mean() for isim, pred in sonuclar.items()}
bars = axes[0].bar(acc_list.keys(), acc_list.values(),
                   color=['steelblue', 'darkorange', 'seagreen'], edgecolor='black')
axes[0].set_ylim(0, 1)
axes[0].set_title('Test Accuracy')
axes[0].tick_params(axis='x', rotation=15)
for bar, v in zip(bars, acc_list.values()):
    axes[0].text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                 f'{v:.3f}', ha='center', fontweight='bold')

# Sağ: CV F1-macro (boxplot — varyansı da gösterir)
f1_data = [cv_sonuclar[m] for m in modeller]
bp = axes[1].boxplot(f1_data, labels=list(modeller.keys()), patch_artist=True)
renkler = ['steelblue', 'darkorange', 'seagreen']
for patch, renk in zip(bp['boxes'], renkler):
    patch.set_facecolor(renk)
axes[1].set_ylim(0, 1)
axes[1].set_title('5-Fold CV F1-macro')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()